# 8. 循环神经网络 (Recurrent Neural Networks, RNN)

> 如果说卷积神经网络（CNN）是深度学习的“双眼”，擅长捕捉**空间**结构；
> 
> 那么循环神经网络（RNN）就是深度学习的“大脑”，擅长处理**时间**序列。

1. **核心矛盾：从“独立”到“关联”**

    - **之前的模型（MLP/CNN）**：假设样本之间是**独立同分布 (IID)** 的。输入一张猫，输出猫；输入下一张狗，与上一张无关。
    - **现实的序列**：在文本、语音、股市中，当前的数值极度依赖于过去。

      - 例子：“我吃了一个____”。（根据前文，这里大概率是“苹果”或“馒头”，而不是“飞机”）。

    - **RNN 的使命**：引入**隐藏状态 (Hidden State)**，让模型具备“记忆”能力。

2. **知识版图：本章的四个核心模块**
  
    - 第一模块：文本预处理 (Text Preprocessing)

        在把文字喂给神经网络前，必须经过洗练：
        1. **词元化 (Tokenization)**：把句子拆成单词或字符。
        2. **词表 (Vocabulary)**：建立“单词”到“数字 ID”的映射。
        3. **加载序列数据**：学习如何通过“随机采样”或“顺序分区”来切分长文本。

     - 第二模块：语言模型 (Language Models)
       
       1. **数学本质**：计算一个序列出现的概率 $P(x_1, x_2, \dots, x_T)$。
       2. **马尔可夫模型 (Markov Models)**：理解 $n$ 元语法 (n-grams) 的局限。
       3. **齐普夫定律 (Zipf's Law)**：理解真实语言中词频分布的极端不平衡（二八定律）。

     - 第三模块：循环神经网络架构 (RNN Architecture)
       
       1. **隐藏状态 ($H_t$)**：理解 $H_t = \phi(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$。
       2. **循环特性**：理解为什么 $W_{hh}$（隐藏层到隐藏层的权重）是实现记忆的关键。
       3. **困惑度 (Perplexity)**：学习评价序列模型好坏的标准（不再仅仅是准确率）。

     - 第四模块：随时间反向传播 (BPTT)
       
       1. **核心痛点**：序列越长，计算图越深。
       2. **梯度危机**：深度理解为什么 RNN 天生容易产生**梯度消失**或**梯度爆炸**。
       3. **截断梯度**：学习工程上如何通过“梯度裁剪”来保证训练不崩溃。

---


## 8.1 序列模型 (Sequence Models)

### 1. 核心数学矛盾：维度的无穷增长

在之前的模型（如 MLP/CNN）中，我们处理的是固定维度的输入。但在序列数据中，为了预测 $t$ 时刻的输出 $x_t$，理论上我们需要考虑之前所有的观测值 $x_1, \dots, x_{t-1}$。

#### 1.1 概率链式法则 (Chain Rule)
任意序列的联合概率分布可以展开为：
$$P(x_1, \dots, x_T) = \prod_{t=1}^T P(x_t \mid x_1, \dots, x_{t-1})$$
- **工程挑战**：随着时间 $t$ 的推移，条件概率的输入向量 $x_1, \dots, x_{t-1}$ 的长度会不断增加，模型无法处理变长的、且趋于无穷大的特征向量。

---

### 2. 解决方案：简化假设

为了使计算可行，序列模型引入了两种核心策略：

#### 2.1 自回归模型 (Autoregressive Models)

假设当前值只取决于过去固定跨度 $\tau$ 内的观测值。
- **数学表达**：$P(x_t \mid x_{t-1}, \dots, x_1) \approx P(x_t \mid x_{t-1}, \dots, x_{t-\tau})$
- **术语**：这被称为 **$\tau$ 阶马尔可夫模型 (Markov Model)**。此时，特征的维度被固定为 $\tau$。

#### 2.2 潜变量模型 (Latent Variable Models)
不保留长长的历史，而是维护一个**隐藏状态 (Hidden State)** $h_t$ 来总结历史。
- **公式**：
    1.  $h_t = g(h_{t-1}, x_{t-1})$ （更新记忆）
    2.  $\hat{x}_t = P(x_t \mid h_t)$ （基于记忆预测）
- **意义**：这是 RNN 的理论基石。

---

### 3. 核心约束：因果性 (Causality)

**因果性**是序列建模的铁律：**预测 $x_t$ 时，模型只能接触到时间轴上 $t$ 以前的数据。**
- **体现**：在构造数据集时，标签必须是特征窗口之后的第一个点。
- **后果**：模型严禁“偷看未来”，任何违反此规则的设计都会导致训练出的模型在实际应用中毫无用处（数据泄露）。

---

### 4. 构造序列数据集

我们将通过预测带噪声的正弦波来演示如何构造符合因果性的训练集。


In [5]:
import torch
from torch import Tensor

class SequenceDataGenerator:
    """序列数据生成与处理器。
    
    负责生成模拟序列数据，并利用滑动窗口技术构造符合因果性约束的特征-标签对。
    """
    def __init__(self, n_points: int = 1000, tau: int = 4):
        """
        Args:
            n_points: 总数据点数。
            tau: 时间窗口大小（马尔可夫阶数）。
        """
        self.n_points = n_points
        self.tau = tau
        self.time = torch.arange(1, n_points + 1, dtype=torch.float32)
        # 生成带噪声的正弦波
        self.x = torch.sin(0.01 * self.time) + torch.normal(0, 0.2, (n_points,))

    def split_data(self, train_percent: float = 0.6) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        """构造滑动窗口数据集并划分训练集与验证集。
        
        因果性体现：X[i] = [x[i], ..., x[i+tau-1]], y[i] = x[i+tau]。
        
        Returns:
            train_features: 形状为 (n_points - tau, tau) 的训练集特征矩阵。
            train_labels: 形状为 (n_points - tau, 1) 的训练集标签向量。
            val_features: 验证集特征矩阵。
            val_labels: 验证集标签向量。
        """
        num_samples = self.n_points - self.tau
        features = torch.zeros((num_samples, self.tau))
        
        # 填充特征矩阵：每一列都是序列的一个偏移版本
        for i in range(self.tau):
            features[:, i] = self.x[i : i + num_samples]
            
        # 标签是紧随窗口后的那个点，即从索引 tau 开始的所有点
        labels = self.x[self.tau:].reshape((-1, 1))
        
        # 按照 train_percent 划分数据集
        n_train = int(num_samples * train_percent)
        train_features = features[:n_train]
        train_labels = labels[:n_train]
        val_features = features[n_train:]
        val_labels = labels[n_train:]
        
        return train_features, train_labels, val_features, val_labels

In [6]:
# --- 验证数据构造 ---
gen = SequenceDataGenerator(tau=4)
train_features, train_labels, _, _ = gen.split_data()
print(f"特征示例 (前4个点): {train_features[0]}")
print(f"标签示例 (第5个点): {train_labels[0]}")

特征示例 (前4个点): tensor([ 0.1014,  0.0633, -0.1796,  0.1595])
标签示例 (第5个点): tensor([-0.1655])



---

### 5. 多步预测 (Multistep Prediction) 的崩溃

这是 8.1 节最重要的结论，揭示了简单自回归模型的局限性。

#### 5.1 单步预测 (One-step-ahead)

-  **定义**：始终使用**真实**的历史数据预测下一个点。
-  **表现**：由于输入是准确的，模型表现通常非常优秀。

#### 5.2 多步预测 (Multistep / K-step-ahead)

-  **定义**：使用模型**自己生成的预测值**作为输入，继续预测更远的未来。
-  **表现**：**误差累积 (Error Accumulation)**。
-  **数学直觉**：如果单步预测有误差 $\epsilon$，在 $k$ 步预测中，误差会按指数级传播。你会发现预测曲线迅速偏离真实轨迹。

---

### 6. 总结 Checklist

-  **自回归策略**：理解如何将变长序列转化为固定长度的特征向量。
-  **因果性约束**：掌握滑动窗口切片时“不准看未来”的索引逻辑。
-  **马尔可夫阶数 ($\tau$)**：明白窗口大小决定了模型“回看”历史的深度。
-  **误差累积**：深刻理解为什么简单的自回归模型无法进行长程预测（多步预测会崩溃）。

---
